# Stochastic Calculus from Zero · Episode 4: Information: filtrations

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The coin world.** A share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses, eight possible worlds. Heads has a 60% real-world chance; a deposit grows 2% a step. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Blocks of information · 4. Known at time n · 5. Price vs history · 6. Four tosses · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 4,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

The spread of the heads fraction goes like $1/\sqrt{n}$, so halving the spread from 100 tosses takes four times as many: 400.

In [ ]:
out["prev"] = {"from": 100, "answer": 400}

## 3. Blocks of information

Before any toss you can't tell any of the 8 worlds apart: one block. After the first toss you know whether it began with H or T: two blocks of four. After two tosses, four blocks of two; after three, every world on its own. What you know at time $n$ is this split of the worlds into blocks, and the blocks only ever split, never merge. The sequence of splits is a **filtration**, written $\mathcal{F}_0, \mathcal{F}_1, \mathcal{F}_2, \mathcal{F}_3$. (Each $\mathcal{F}_n$ is what mathematicians call a sigma-algebra: every event you can decide by time $n$, which here means every union of blocks.)

In [ ]:
cw = CoinWorld()
rows = cw.table()
blocks = {n: cw.blocks(n) for n in range(cw.steps + 1)}
counts = {str(n): len(b) for n, b in blocks.items()}
assert list(counts.values()) == [1, 2, 4, 8]
# the blocks only split: every block at n + 1 sits inside one block at n
for n in range(cw.steps):
    assert all(any(set(b) <= set(a) for a in blocks[n]) for b in blocks[n + 1])
out["coin"] = {"worlds": rows}
out["blocks"] = {"counts": counts, "n0": 1, "n1": 2, "n2": 4, "n3": 8}
counts

## 4. Known at time n

An event is **known at time $n$** if, once you know which block you are in, you can say for sure whether it happened: the event is a union of whole blocks. "The first toss was heads" is known at time 1. "The share ends above 100" cuts through blocks until time 3.

In [ ]:
def known_at(event, n):
    '''True if the event (a set of worlds) is a union of blocks at time n.'''
    return all(set(b) <= event or not (set(b) & event) for b in cw.blocks(n))

first_heads = {w for w in cw.worlds if w[0] == "H"}
above = {w for w in cw.worlds if cw.price(w, 3) > 100}
out["known"] = {"first_heads": sorted(first_heads), "above_100": sorted(above),
                "first_heads_at": min(n for n in range(4) if known_at(first_heads, n)),
                "above_at": min(n for n in range(4) if known_at(above, n))}
assert out["known"]["first_heads_at"] == 1 and out["known"]["above_at"] == 3
out["known"]

## 5. Price vs history

After HT and after TH the share is at the same price: $100 \times 1.2 \times 0.8 = 96$. So the price at time 2 splits the worlds into only three groups (144, 96, 64), while the history splits them into four. Knowing the price is knowing **less** than knowing the history. In symbols: $\sigma(S_2)$ is coarser than $\mathcal{F}_2$.

In [ ]:
price_groups = sorted({round(cw.price(w, 2), 9) for w in cw.worlds}, reverse=True)
mid = [w for w in cw.worlds if abs(cw.price(w, 2) - 96) < 1e-9]
out["price_vs_history"] = {"price_groups": price_groups, "n_price_groups": len(price_groups), "n_history": counts["2"],
                           "mid_price": cw.price("HT", 2), "mid_worlds": mid,
                           "groups": [[w for w in cw.worlds if abs(cw.price(w, 2) - x) < 1e-9] for x in price_groups]}
assert len(price_groups) == 3 and abs(cw.price("HT", 2) - cw.price("TH", 2)) < 1e-12
out["price_vs_history"]

## 6. Four tosses

With four tosses there are $2^4 = 16$ histories, but the price only depends on the number of heads (0 to 4): 5 different prices.

In [ ]:
cw4 = CoinWorld(steps=4)
out["four"] = {"histories": len(cw4.worlds), "prices": len({round(cw4.price(w, 4), 9) for w in cw4.worlds})}
assert out["four"] == {"histories": 16, "prices": 5}

## 7. This episode's question

The same four-toss count is the question: 16 histories after the last toss, and 5 different prices.

In [ ]:
out["question"] = {"tosses": 4} | out["four"]

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** List the blocks of information made by "number of heads so far" at time 2. How many are there, and how do they compare with $\mathcal{F}_2$?

<details><summary>Answer</summary>

Three groups: {HH·} (2 heads), {HT·, TH·} (1 head), {TT·} (0 heads). It is coarser than $\mathcal{F}_2$, which has four blocks: like the price, the count of heads forgets the order.

</details>

**Exercise 2 (By hand).** Is the event "$S_1 > S_2$" known at time 2? At time 1?

<details><summary>Answer</summary>

$S_1 > S_2$ means the second toss was tails: worlds HT· and TT·. Those are whole blocks at time 2, so it is known at time 2. At time 1 the block H·· contains both HH· and HT·, so it is not known yet.

</details>

**Exercise 3 (By hand).** With four tosses, how many blocks are there at times 0, 1, 2, 3, 4?

<details><summary>Answer</summary>

1, 2, 4, 8, 16: each toss splits every block in two.

</details>

**Exercise 4 (In code).** Use `known_at` to find, for every event "the share ends at price x", the first time it is known.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
first = {}
for x in sorted({round(cw.price(w, 3), 6) for w in cw.worlds}, reverse=True):
    ev = {w for w in cw.worlds if abs(cw.price(w, 3) - x) < 1e-9}
    first[x] = min(n for n in range(4) if known_at(ev, n))
assert first[172.8] == 3 and first[51.2] == 3
print(first)

**Exercise 5 (In code).** How many blocks does the final price $S_3$ alone make? Check it is coarser than $\mathcal{F}_3$.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
groups = {}
for w in cw.worlds:
    groups.setdefault(round(cw.price(w, 3), 9), []).append(w)
assert len(groups) == 4 < len(cw.blocks(3))
print({k: v for k, v in groups.items()})

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")